In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from functions import *

In [ ]:
OWL_PATH1 = "data/oeo-full.owl"
OWL_PATH2 = "data/go-plus.owl"

In [ ]:
base_cfg = BoxConfig(
    dim = 6,
    steps = 10000,
    seed = 42,
    size_weight = 0.1,
    subclass_weight = 5.0,
    disjoint_weight = 1.0,
    big_box_weight = 0.1,
    depth_scale = 0.5,
    distance_weight = 0.1)

In [ ]:
schedule=CurriculumSchedule(
    subclass_start = 0.0,   # structural foundation first
    disjoint_start = 0.4,   # separation only after containment exists
    sibling_start = 0.5,
    big_box_start = 0.7,   # size control
    ramp = True
)

In [ ]:
results_plain_oeo = sweep_dimensions(
    owl_path = OWL_PATH1,
    learn_fn = learn_boxes_from_owl,
    dims = range(2, 21),
    cfg = base_cfg,
    path = "saved1/plain_oeo",
)


In [ ]:
results_curriculum_oeo = sweep_dimensions(
    owl_path = OWL_PATH1,
    learn_fn = learn_boxes_with_curriculum,
    schedule = schedule,
    dims = range(2, 21),
    cfg = base_cfg,
    path = "saved1/curr_oeo",
)

In [ ]:
results_plain_oeo_noise = sweep_dimensions(
    owl_path = OWL_PATH1,
    learn_fn = learn_boxes_from_owl,
    dims = range(2, 21),
    cfg = base_cfg,
    noise = True,
    path = "saved1/plain_oeo_noise",
)

In [ ]:
results_curriculum_oeo_noise = sweep_dimensions(
    owl_path = OWL_PATH1,
    learn_fn = learn_boxes_with_curriculum,
    schedule = schedule,
    dims = range(2, 21),
    cfg = base_cfg,
    noise = True,
    path = "saved1/curr_oeo_noise",
)

In [ ]:
results_plain_go = sweep_dimensions(
    owl_path = OWL_PATH2,
    learn_fn = learn_boxes_from_owl,
    dims = range(5, 26),
    cfg = base_cfg,
    path = "saved1/plain_go",
)

In [ ]:
results_plain_go_noise = sweep_dimensions(
    owl_path = OWL_PATH2,
    learn_fn = learn_boxes_from_owl,
    dims = range(5, 26),
    cfg = base_cfg,
    noise = True,
    path = "saved1/plain_go_noise",
)

In [ ]:
results_curriculum_go = sweep_dimensions(
    owl_path = OWL_PATH2,
    learn_fn = learn_boxes_with_curriculum,
    schedule = schedule,
    dims = range(5, 26),
    cfg = base_cfg,
    path = "saved1/curr_go",
)

In [ ]:
results_curriculum_go_noise = sweep_dimensions(
    owl_path = OWL_PATH2,
    learn_fn = learn_boxes_with_curriculum,
    schedule = schedule,
    dims = range(5, 26),
    cfg = base_cfg,
    noise = True,
    path = "saved1/curr_go_noise",
)

careful how to call it, must use load_owl or load_owl_with errors accordingly!

In [ ]:
classes, subclass_of, disjoint_pairs = load_owl(OWL_PATH1)
results_plain_oeo = load_sweep_results("saved1/plain_oeo", classes, subclass_of, disjoint_pairs)
results_curriculum_oeo = load_sweep_results("saved1/curr_oeo", classes, subclass_of, disjoint_pairs)

In [ ]:
classes, subclass_of, disjoint_pairs = load_owl_with_errors(OWL_PATH1)
results_plain_oeo_noise = load_sweep_results("saved1/plain_oeo_noise", classes, subclass_of, disjoint_pairs)
results_curriculum_oeo_noise = load_sweep_results("saved1/curr_oeo_noise", classes, subclass_of, disjoint_pairs)

### eval

In [ ]:
plot_sweep_comparison(results_plain_oeo, results_curriculum_oeo, "OEO")

In [ ]:
plot_sweep_comparison(results_plain_oeo_noise, results_curriculum_oeo_noise, "OEO Noise")

In [ ]:
eval_plain_oeo = evaluate_models(results_plain_oeo)
plot_evaluation(eval_plain_oeo, title="Plain Box Embedding — OEO")

eval_curriculum_oeo = evaluate_models(results_curriculum_oeo)
plot_evaluation(eval_curriculum_oeo, title="Curriculum Box Embedding — OEO")


In [ ]:
eval_plain_oeo_noise = evaluate_models(results_plain_oeo_noise)
plot_evaluation(eval_plain_oeo_noise, title="Plain Box Embedding — OEO Noise")

eval_curriculum_oeo_noise = evaluate_models(results_curriculum_oeo_noise)
plot_evaluation(eval_curriculum_oeo_noise, title="Curriculum Box Embedding — OEO Noise")


In [ ]:
eval_conc = evaluate_concluded_relationships(results_plain_oeo, disjoint_margin=0.05)
plot_concluded_evaluation(eval_conc, "OEO – Concluded Relationships (Plain)")

In [ ]:
eval_conc = evaluate_concluded_relationships(results_plain_oeo_noise, disjoint_margin=0.05)
plot_concluded_evaluation(eval_conc, "OEO – Concluded Relationships (Plain - Noise)")

In [ ]:
eval_conc_curr = evaluate_concluded_relationships(results_curriculum_oeo, disjoint_margin=0.05)
print(eval_conc_curr)
plot_concluded_evaluation(eval_conc_curr, "OEO – Concluded Relationships (Curriculum)")

In [ ]:
eval_conc_curr = evaluate_concluded_relationships(results_curriculum_oeo_noise, disjoint_margin=0.05)
plot_concluded_evaluation(eval_conc_curr, "OEO – Concluded Relationships (Curriculum Noise)")

In [ ]:
results = sweep_schedule_combinations(
    owl_path=OWL_PATH1,
    params={
        "disjoint_start" : [0.0, 0.5],
        "subclass_start" : [0.0, 0.5],
        "big_box_start"  : [0.0, 0.5],
        "sibling_start"  : [0.0, 0.5],
    },
    dim=4,
    steps=5000,
    base_schedule=CurriculumSchedule(ramp=True),
)
plot_combo_heatmaps(results)
plot_combo_lines(results, x_param="disjoint_start",
                 color_param="subclass_start", metric="sub_viol")

In [ ]:
results = sweep_schedule_combinations(
    owl_path=OWL_PATH2,
    params={
        "disjoint_start" : [0.0, 0.5],
        "subclass_start" : [0.0, 0.5],
        "big_box_start"  : [0.0, 0.5],
        "sibling_start"  : [0.0, 0.5],
    },
    dim=4,
    steps=1000,
    base_schedule=CurriculumSchedule(ramp=True),
)
plot_combo_heatmaps(results, metrics=["sub_viol", "dis_viol", "loss_final", ""])
plot_combo_lines(results, x_param="disjoint_start",
                 color_param="subclass_start", metric="sub_viol")

In [ ]:
plot_all_starts_heatmap(results, title="All Start Parameters — Marginal Effects")